# 📝 Neo4j 시작하기 과제 LV3(통합): 그래프 모델 설계

> 지금까지 배운 노드·관계·속성 모델을 **직접 설계**하고, 그 설계로 실제 질문에 답할 수 있는지 확인합니다. 정답은 하나가 아닙니다. **레이블 이름은 자유**이고, **원칙**만 지키면 됩니다.

## 풀이 방법
1. 준비 셀 → 점검 셀 → 검증기 두 셀을 먼저 실행하세요.
2. **1번**: 서점 도메인을 markdown 으로 설계한 뒤, 파이썬 dict 로 옮겨 `validate_model` 로 확인합니다.
3. **2번**: 그 모델로 네 가지 질문에 답할 수 있는지 `check_reachable` 로 확인합니다.
4. **3번**: 후보 모델 5개에 검증기를 돌려 걸리는 것을 찾고, 검증기가 못 잡는 결함 하나는 서술합니다.
5. **4번**(서술형): Movies 모델 확장을 판단해 서술합니다.

화이팅!

아래 준비 셀·점검 셀·검증기 셀을 먼저 실행하세요.

In [ ]:
# Neo4j 연결: 이 셀은 실행만 하세요(내용은 이해하지 않아도 됩니다).
# - .env 의 NEO4J_URI/NEO4J_USER/NEO4J_PASSWORD 로 데이터베이스에 연결합니다.
# - run_cypher("쿼리", 파라미터=값) 가 결과를 dict 리스트로 돌려줍니다. 이 헬퍼로 Cypher 를 실행합니다.
# - 이 단원은 그래프를 조회만 합니다(그래프를 바꾸지 않습니다).
import os

from dotenv import load_dotenv
from neo4j import GraphDatabase   # 파이썬용 공식 드라이버. 이 클래스로 접속 통로를 연다

# 1) 접속 정보 읽기: .env 에 적힌 값을 환경변수로 올린다(파일이 없으면 조용히 넘어간다)
load_dotenv(".env")       # 같은 폴더의 .env
load_dotenv("../.env")    # 정답 폴더에서 실행하는 경우

# 두 번째 인자는 .env 에 그 키가 없을 때 쓰는 기본값이다(로컬 Desktop 의 표준 주소·사용자).
# .env 를 못 읽어도 에러가 아니라 이 값으로 조용히 넘어가니, 이 셀 마지막 줄에 찍히는
# 주소가 실습 전용 DB 가 맞는지 눈으로 꼭 확인한다
NEO4J_URI = os.getenv("NEO4J_URI", "bolt://localhost:7687")
NEO4J_USER = os.getenv("NEO4J_USER", "neo4j")
NEO4J_PASSWORD = os.getenv("NEO4J_PASSWORD", "neo4j")
# 2) 드라이버 만들기: 접속 통로 하나를 노트북 전체가 나눠 쓴다(쿼리마다 새로 만들지 않는다)
driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD))
driver.verify_connectivity()   # 실제로 붙어 본다. 인스턴스가 꺼져 있거나 비밀번호가 틀리면 여기서 에러가 난다


# 3) 수업 내내 쓰는 헬퍼: 쿼리를 보내고 결과를 파이썬 자료형으로 바꿔 준다
def run_cypher(query, **params):
    """Cypher 실행 -> 결과를 dict 리스트로 반환(수업 공용 헬퍼)."""
    with driver.session() as session:
        # 세션은 with 블록을 벗어나면 자동으로 닫힌다. record.data() 가 결과 한 행을 dict 로 바꾼다
        return [record.data() for record in session.run(query, **params)]


print("Neo4j 연결:", NEO4J_URI)   # 이 줄이 찍히면 연결까지 성공한 것이다

In [ ]:
# Movies 그래프가 적재돼 있는지 점검: 실행만 하세요(그래프를 바꾸지 않습니다).
# MATCH (n) 은 레이블을 가리지 않고 모든 노드를 고른다. count(n) 결과는 한 행이라 [0] 으로 dict 를 꺼낸다
_n = run_cypher("MATCH (n) RETURN count(n) AS cnt")[0]["cnt"]   # 이름 앞 밑줄은 이 셀에서만 쓰는 임시 변수라는 표시
print("연결된 그래프의 노드 수:", _n)   # 171 이면 준비 완료, 0 이면 아직 적재 전이다

In [ ]:
# 모델 검증기(실행만 하세요). 여러분이 설계한 모델을 규칙에 비춰 봅니다.
QUESTIONS = ['같은 카테고리 책', '이 저자의 다른 책', '이 독자가 산 책', '이 책 평균 별점']


def validate_model(model):
    """model = {
        'nodes': {레이블...},
        'relationships': {관계이름: (주어레이블, 목적어레이블)},
        'node_properties': {레이블: {속성이름...}},
        'rel_properties': {관계이름: {속성이름...}},
    }
    레이블·관계 **이름은 자유**입니다. 아래 규칙만 봅니다.
    1) 노드 레이블 5개 이상, 관계 4개 이상
    2) 관계 이름은 대문자 스네이크, 주어·목적어 레이블은 nodes 안에 있어야 함
    3) 어떤 관계에도 연결되지 않은 레이블이 없어야 함(끊긴 노드 금지)
    4) 별점(rating)은 **관계 속성**이어야 하고, 노드 속성이면 안 됨
    """
    nodes = model.get('nodes', set())
    rels = model.get('relationships', {})
    node_props = model.get('node_properties', {})
    rel_props = model.get('rel_properties', {})
    assert isinstance(nodes, (set, frozenset)), (
        "nodes 는 중괄호로 감싼 집합이어야 합니다: {'Reader', 'Book', ...} "
        '(대괄호 리스트로 적으면 여기서 걸립니다)')
    assert isinstance(rels, dict), (
        'relationships 는 {관계이름: (주어레이블, 목적어레이블), ...} 형태의 dict 여야 합니다')
    assert len(nodes) >= 5, '시나리오에 등장하는 개체를 빠짐없이 노드로 옮기세요(레이블 5개 이상)'
    assert len(rels) >= 4, '관계를 4개 이상 설계하세요'
    for name, endpoints in rels.items():
        assert name.replace('_', '').isupper(), f'관계 이름은 대문자 스네이크로: {name}'
        subj, obj = endpoints
        assert subj in nodes, f'{name} 의 주어 레이블 {subj} 가 nodes 에 없습니다'
        assert obj in nodes, f'{name} 의 목적어 레이블 {obj} 가 nodes 에 없습니다'
    used = {label for pair in rels.values() for label in pair}
    orphans = nodes - used
    assert not orphans, f'어떤 관계에도 연결되지 않은 레이블이 있습니다: {sorted(orphans)}'
    for label in node_props:
        assert label in nodes, f'node_properties 의 {label} 가 nodes 에 없습니다'
    for name in rel_props:
        assert name in rels, f'rel_properties 의 {name} 가 relationships 에 없습니다'
    assert any('rating' in props for props in rel_props.values()), \
        "별점(rating)은 '누가 무엇을 평가했나'마다 달라지는 값입니다 - 관계 속성으로 두세요"\
        " (별점 자체를 기준으로 다른 개체와 연결할 일이 없다면 노드로 올리지 않습니다)"
    assert not any('rating' in props for props in node_props.values()), \
        '별점(rating)을 노드 속성으로 두면 한 대상에 하나밖에 담기지 않습니다'
    return True


In [ ]:
# 도달성 검사(실행만 하세요). 네 질문을 모델 안에서 실제로 따라갈 수 있는지 봅니다.
def _walk_ends(rels, chain):
    """관계 이름을 차례로 따라 걸었을 때 (시작 레이블, 끝 레이블, 거쳐간 레이블들) 목록."""
    results = []
    for start in set(rels[chain[0]]):
        current, path, ok = start, [start], True
        for name in chain:
            left, right = rels[name]
            if current == left:
                current = right
            elif current == right:
                current = left
            else:
                ok = False
                break
            path.append(current)
        if ok:
            results.append((start, current, tuple(path)))
    return results


def check_reachable(model, answers):
    """answers = {질문: [관계이름, ...]}. 그 질문에 답하려고 따라갈 관계들을 순서대로."""
    rels = model.get('relationships', {})
    rel_props = model.get('rel_properties', {})
    missing = [q for q in QUESTIONS if q not in answers]
    assert not missing, f'답하지 않은 질문이 있습니다: {missing}'
    walks = {}
    for question in QUESTIONS:
        chain = answers[question]
        assert isinstance(chain, (list, tuple)) and chain, f"'{question}' 의 답은 관계 이름 목록이어야 합니다"
        for name in chain:
            assert name in rels, f"'{question}': 모델에 없는 관계 이름 {name}"
        ends = _walk_ends(rels, chain)
        assert ends, f"'{question}': 관계들이 서로 이어지지 않습니다 - 모델 안에서 따라갈 수 없는 경로입니다"
        walks[question] = ends

    def round_trip(question):
        assert len(answers[question]) == 2, (
            f"'{question}': 거쳐 갔다가 되돌아오는 길이라 관계 이름을 두 개 적어야 합니다"
            '(책 -> 거쳐 가는 것 -> 책). 같은 관계를 두 번 적어도 됩니다')
        found = [w for w in walks[question] if w[0] == w[1] and w[2][1] != w[0]]
        assert found, (f"'{question}': 출발한 곳으로 되돌아오지 못했습니다 - "
                       '기준이 되는 값을 노드로 두지 않으면 이 길이 만들어지지 않습니다')
        return found

    by_category = round_trip(QUESTIONS[0])
    by_author = round_trip(QUESTIONS[1])
    assert {w[2][1] for w in by_category} != {w[2][1] for w in by_author}, \
        '앞의 두 질문은 서로 다른 레이블을 거쳐 돌아와야 합니다'
    assert list(answers[QUESTIONS[0]]) != list(answers[QUESTIONS[1]]), \
        '앞의 두 질문에 같은 답을 적었습니다 - 거쳐 가는 것이 서로 달라야 합니다'
    direct = [w for w in walks[QUESTIONS[2]] if len(answers[QUESTIONS[2]]) == 1 and w[0] != w[1]]
    assert direct, f"'{QUESTIONS[2]}': 두 레이블을 바로 잇는 관계 하나면 됩니다"
    buy_rel = answers[QUESTIONS[2]][0]
    assert 'rating' not in rel_props.get(buy_rel, set()), \
        (f"'{QUESTIONS[2]}': {buy_rel} 는 별점이 붙은 관계입니다 - "
         '구매와 평가는 다른 사실이니 다른 관계로 답하세요')
    rating_chain = answers[QUESTIONS[3]]
    assert len(rating_chain) == 1, f"'{QUESTIONS[3]}': 관계 하나로 답할 수 있어야 합니다"
    assert 'rating' in rel_props.get(rating_chain[0], set()), \
        f"'{QUESTIONS[3]}': 그 관계에 별점(rating) 속성이 없습니다"
    common = set.intersection(*[{label for w in walks[q] for label in w[2]} for q in QUESTIONS])
    assert common, '네 질문이 공통으로 지나는 레이블(책)이 없습니다'
    return True


## 1. 온라인 서점 그래프 모델 설계하기
**배경**: 새 서비스를 그래프로 옮기는 첫 단계는 **무엇을 노드·관계·속성으로 둘지** 정하는 것입니다. 아래 시나리오를 읽고 그래프 모델을 직접 설계합니다.

**시나리오(온라인 서점)**
- 독자가 회원으로 가입합니다. 독자는 **책을 구매**하고, 책에 **리뷰(별점)를 남깁니다**.
- 책은 **저자가 집필**했고, 한 **출판사가 출간**했습니다.
- 책은 하나의 **카테고리**(예: 소설·경제)에 속합니다.
- 자주 던지는 질문: "같은 카테고리의 책", "이 저자의 다른 책", "이 독자가 구매한 책", "이 책의 평균 별점".

**요구사항**
1. 먼저 **아래 markdown 셀**에 설계를 표로 적으세요. 노드 레이블, 관계(이름·방향·주어/목적어), 그리고 **어디에 어떤 속성**을 둘지(특히 **별점**을 어디에 둘지).
2. 그다음 **답안 셀**에서 설계를 파이썬 dict **`book_model`** 로 옮기고 `validate_model(book_model)` 로 검증하세요.

**`book_model` 형식** (검증기 docstring 과 동일)

```text
{
  'nodes': {레이블, ...},
  'relationships': {관계이름: (주어레이블, 목적어레이블), ...},
  'node_properties': {레이블: {속성이름, ...}, ...},
  'rel_properties': {관계이름: {속성이름, ...}, ...},
}
```

`nodes` 는 **집합**(중괄호)으로, 나머지 셋은 **dict** 로 적습니다. 속성 목록도 집합입니다.

**결정 규칙(채점 기준)**
- 레이블은 **명사**(예: `Reader`), 관계는 **대문자 스네이크**(예: `PURCHASED`). **레이블 이름은 자유롭게 지어도 됩니다**(한글이어도 통과합니다). 관계 이름은 **영문 대문자 스네이크**로 쓰세요.
- 시나리오에 나오는 개체를 빠짐없이 옮기면 **레이블 5개 이상·관계 4개 이상**이 됩니다(검증기가 이 수를 확인합니다).
- 모든 관계의 주어·목적어 레이블은 `nodes` 안에 있어야 하고, **어떤 관계에도 연결되지 않은 레이블이 있으면 안 됩니다**.
- 별점은 속성 이름을 **`rating`** 으로 쓰세요(검증기가 그 이름을 찾습니다). **어디에 둘지는 "그 값이 무엇마다 달라지는가" 로 판단**하세요(검증기도 그 원칙으로 봅니다).
- 그 밖에 어떤 개체·관계를 둘지는 시나리오를 읽고 스스로 정합니다(명사는 노드, 행위는 관계).

> 한 책을 여러 독자가 **서로 다른 별점**으로 평가한다는 점을 떠올리세요(Movies 의 `rating` 이 어디에 붙어 있었는지도 힌트가 됩니다).

*(여기에 설계 표를 작성하세요. 노드 / 관계(방향) / 속성 위치)*

| 구분 | 이름 | 설명·속성 |
|---|---|---|
|  |  |  |

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점] 여기서 보는 것은 형식과 원칙뿐입니다(설계가 쓸모 있는지는 2번에서 확인).
assert validate_model(book_model) is True
print('✅ 형식 검사 통과! 이 설계로 실제 질문에 답할 수 있는지는 2번에서 확인합니다.')

## 2. 설계한 모델로 질문에 답할 수 있는가
**배경**: 모델이 좋은지 아닌지는 **자주 던지는 질문에 답할 수 있는가**로 갈립니다. 1번에서 만든 `book_model` 로 네 질문에 실제로 답할 수 있는지 확인합니다.

**요구사항**
- 각 질문에 답하려면 **어떤 관계를 어떤 순서로 따라가야 하는지**를 관계 이름 리스트로 적어 dict **`answers`** 에 담으세요. 키는 `QUESTIONS` 의 네 문자열을 **그대로** 씁니다.
- 그다음 `check_reachable(book_model, answers)` 로 확인하세요.

```text
answers = {
    '같은 카테고리 책': ['관계이름', '관계이름'],   # 거쳐 갔다가 돌아오므로 관계 이름 두 개
    '이 저자의 다른 책': ['관계이름', '관계이름'],   # 같은 관계를 두 번 적어도 된다
    '이 독자가 산 책': ['관계이름'],               # 바로 잇는 관계 하나
    '이 책 평균 별점': ['관계이름'],               # 별점이 붙어 있는 관계 하나
}
```

**생각할 거리**: "같은 카테고리의 책"은 **한 책에서 출발해 무언가를 거쳐 다시 책으로 돌아오는** 길입니다. 여러분의 모델에 그 길이 실제로 있나요? 없다면 1번 설계로 돌아가 고치세요. 그게 이 문제의 핵심입니다.

<details><summary>힌트</summary>

```text
접근방법:
- 질문마다 출발 레이블과 도착 레이블을 먼저 정하고, 그 사이를 잇는 관계를 순서대로 적는다.

세부구현:
1. 한 책에서 같은 무리의 다른 책으로 가려면 무엇을 거쳐야 하는지 생각한다.
   1-1. 거쳐 가는 그것이 모델에 노드로 있는지 확인한다.
2. 독자와 책을 바로 잇는 관계는 하나면 충분하다.
3. 평균 별점은 별점 속성이 붙어 있는 그 관계 하나로 답한다.
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
assert check_reachable(book_model, answers) is True
print('✅ 통과!')

## 3. 후보 모델 점검하기
**배경**: 남이 만든 모델을 읽고 결함을 짚는 것도 설계 능력입니다. 아래 제공 셀이 같은 서점 시나리오에 대한 후보 모델 **5개**(A~E)와, 모델 하나가 검증기를 통과하는지 알려 주는 `model_error(model)` 을 줍니다.

**요구사항**:
- `candidates` 의 각 후보에 **`model_error` 를 돌려**, 검증기에 **걸린 후보만** `{알파벳: 검증기 메시지}` 형태의 dict 로 변수 **`caught`** 에 담으세요(`model_error` 는 통과하면 `None`, 걸리면 그 사유를 문자열로 돌려줍니다).
- 메시지는 **검증기가 낸 문장 그대로** 담습니다. 그래야 어느 규칙에 걸렸는지가 남습니다.

**이어서(서술)**: 검증기를 **통과했지만 설계가 잘못된 후보가 하나** 있습니다. 다음 markdown 셀에 어느 후보이고, 무엇이 잘못됐으며, **왜 검증기가 그것을 못 잡는지** 서술하세요(이 부분은 자가채점이 없습니다. 정답 노트북의 모범 서술과 비교하세요).

**판단 기준**: 지금까지 배운 세 가지를 확인하세요.
- 별점처럼 **평가마다 달라지는 값**이 엉뚱한 자리에 있지 않은가
- 시나리오의 개체가 **연결되지 않고 떠 있지** 않은가
- **관계의 방향**이 사실과 맞는가(관계 이름을 소리 내어 읽어 보면 드러납니다. 누가 무엇을 했나)

> 레이블·관계 **이름이 다른 것은 결함이 아닙니다**(1번에서 본 대로 이름은 자유).

**자가 점검**: 서술을 쓴 뒤 스스로 확인하세요.
- [ ] 지목한 후보의 **어느 관계**가 문제인지 이름을 적었다.
- [ ] 그 관계를 **말로 읽었을 때** 무엇이 이상해지는지 한 문장으로 썼다.
- [ ] **검증기가 못 잡는 이유**(도구가 볼 수 없는 것이 무엇인지)를 밝혔다.

In [ ]:
# 후보 모델 5개(실행만 하세요). 내용을 읽고 3번 문제를 푸세요.
candidates = {
    'A': {
        'nodes': {'Reader', 'Book', 'Author', 'Publisher', 'Category'},
        'relationships': {'PURCHASED': ('Reader', 'Book'), 'REVIEWED': ('Reader', 'Book'),
                          'WROTE': ('Author', 'Book'), 'PUBLISHED': ('Publisher', 'Book'),
                          'IN_CATEGORY': ('Book', 'Category')},
        'node_properties': {'Book': {'title', 'isbn'}, 'Reader': {'name'}},
        'rel_properties': {'REVIEWED': {'rating'}},
    },
    'B': {
        'nodes': {'Reader', 'Book', 'Author', 'Publisher', 'Category'},
        'relationships': {'PURCHASED': ('Reader', 'Book'), 'REVIEWED': ('Reader', 'Book'),
                          'WROTE': ('Author', 'Book'), 'PUBLISHED': ('Publisher', 'Book'),
                          'IN_CATEGORY': ('Book', 'Category')},
        'node_properties': {'Book': {'title', 'rating'}},
        'rel_properties': {},
    },
    'C': {
        'nodes': {'Reader', 'Book', 'Author', 'Publisher', 'Category'},
        'relationships': {'PURCHASED': ('Reader', 'Book'), 'REVIEWED': ('Reader', 'Book'),
                          'WROTE': ('Book', 'Author'), 'PUBLISHED': ('Publisher', 'Book'),
                          'IN_CATEGORY': ('Book', 'Category')},
        'node_properties': {'Book': {'title'}},
        'rel_properties': {'REVIEWED': {'rating'}},
    },
    'D': {
        'nodes': {'Reader', 'Book', 'Author', 'Publisher', 'Category'},
        'relationships': {'PURCHASED': ('Reader', 'Book'), 'REVIEWED': ('Reader', 'Book'),
                          'WROTE': ('Author', 'Book'), 'IN_CATEGORY': ('Book', 'Category')},
        'node_properties': {'Book': {'title'}, 'Publisher': {'name'}},
        'rel_properties': {'REVIEWED': {'rating'}},
    },
    'E': {
        'nodes': {'회원', '도서', '작가', '펴낸곳', '분야'},
        'relationships': {'BOUGHT': ('회원', '도서'), 'RATED': ('회원', '도서'),
                          'AUTHORED': ('작가', '도서'), 'ISSUED': ('펴낸곳', '도서'),
                          'BELONGS_TO': ('도서', '분야')},
        'node_properties': {'도서': {'title'}},
        'rel_properties': {'RATED': {'rating'}, 'BOUGHT': {'bought_at'}},
    },
}
def model_error(model):
    """검증기를 통과하면 None, 걸리면 그 사유(메시지 문자열)를 돌려준다."""
    try:
        validate_model(model)
        return None
    except AssertionError as _exc:
        return str(_exc)


for _key in sorted(candidates):
    _model = candidates[_key]
    print(_key, '| 노드:', sorted(_model['nodes']))
    print('   관계:', {_name: _pair for _name, _pair in _model['relationships'].items()})
    print('   노드 속성:', _model['node_properties'], '| 관계 속성:', _model['rel_properties'])


In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점] 후보마다 검증기를 다시 돌려 대조합니다(알파벳만 적어 넣으면 통과하지 않습니다).
_expected = {_k: model_error(_m) for _k, _m in candidates.items() if model_error(_m) is not None}
assert type(caught) is dict, 'caught 는 {알파벳: 검증기 메시지} 형태의 dict 여야 합니다'
assert set(caught) == set(_expected), \
    '검증기에 걸린 후보 목록이 다릅니다. 후보 다섯 개에 각각 model_error 를 돌려 보세요'
assert caught == _expected, \
    '메시지가 다릅니다. model_error 가 돌려준 문장을 그대로(줄이거나 바꾸지 말고) 담으세요'
print('✅ 통과! 걸린 후보:', sorted(caught))

*(여기에 서술하세요. 검증기를 통과했지만 잘못된 후보는?)*

- 후보: 
- 무엇이 잘못됐나: 
- 검증기가 못 잡는 이유: 

## 4. (서술형) Movies 모델 확장: 촬영 국가와 관람 시각은 어디에?
**배경**: Movies 그래프에 두 정보를 새로 담으려 합니다. 이 문제는 **자가채점이 없습니다**. markdown 셀에 서술하고 정답 노트북과 비교하세요.

**요구사항**: 다음 두 가지를 각각 **노드로 둘지, 노드 속성으로 둘지, 관계 속성으로 둘지** 판단하고 **근거**를 서술하세요. 담을 자리를 만들려면 **새 관계가 필요한지**도 함께 밝히세요.

1. **촬영 국가**(예: 미국·프랑스). 한 영화가 어느 나라에서 찍혔는지를 담고, "같은 나라에서 찍은 영화"를 자주 조회할 예정입니다.
2. **관람 시각**(어떤 사용자가 그 영화를 **언제 봤는지**). 나중에 "이 사람이 지난달에 본 영화"를 묻고 싶습니다.

**판단 기준(힌트)**: "그 값을 기준으로 자주 **조회·연결**하는가", "여러 대상이 그 값을 **나누어 갖는가**", "**주체마다 값이 달라지는가**". 교안_02 4절의 판단 트리를 새 대상에 그대로 적용해 보세요.

**자가 점검**: 답을 쓴 뒤 스스로 확인하세요.
- [ ] 두 항목 모두 **노드 / 노드 속성 / 관계 속성** 중 하나를 분명히 골랐다.
- [ ] 각 근거가 위 판단 기준의 **어느 물음에 답한 것인지** 드러난다.
- [ ] 그 값을 담으려면 Movies 의 **지금 있는 관계로 충분한지** 따져 보고 밝혔다.

*(여기에 자신의 답을 서술하세요)*

1. 촬영 국가 → (노드 / 노드 속성 / 관계 속성) · 근거: 
2. 관람 시각 → (노드 / 노드 속성 / 관계 속성) · 근거: 

---
수고했어요! LV3 에서 새 도메인(서점)의 그래프 모델을 **직접 설계**하고, 그 모델로 실제 질문에 답할 수 있는지 확인했으며, 남의 모델에서 **결함을 짚고**, Movies 확장을 판단했습니다. 다음 단원부터는 이 모델을 **Cypher 로 직접 만들고 조회**합니다.